In [1]:

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

T, N, SCALE = 20, 2000, 10.0

# Dataset
def step(s, a):
    s = s + 0.2 * a
    return s, -np.abs(s)

s = np.random.uniform(-1, 1, N)
q = np.random.rand(N)
S, A, R = [np.zeros((N,T), np.float32) for _ in range(3)]

for t in range(T):
    a = (1-q)*(-np.clip(s/0.2,-1,1)) + q*np.random.uniform(-1,1,N)
    S[:,t], A[:,t] = s, a
    s, R[:,t] = step(s,a)

RTG = np.cumsum(R[:,::-1],1)[:,::-1].copy()
S, A, G = [torch.tensor(x)[...,None] for x in (S,A,RTG/SCALE)]
time = torch.arange(T)

# Attention
class Attention(nn.Module):
    def __init__(self,d,h):
        super().__init__()
        self.h = h
        self.qkv = nn.Linear(d,3*d)
        self.proj = nn.Linear(d,d)
        self.register_buffer("mask",torch.tril(torch.ones(3*T,3*T)).bool())

    def forward(self,x):
        B,L,D = x.shape
        q,k,v = [z.view(B,L,self.h,D//self.h).transpose(1,2)
                 for z in self.qkv(x).split(D,-1)]
        a = (q @ k.transpose(-2,-1))/((D//self.h)**0.5)
        a = F.softmax(a.masked_fill(~self.mask[:L,:L],float('-inf')),-1)
        return self.proj((a @ v).transpose(1,2).reshape(B,L,D))

class Block(nn.Module):
    def __init__(self,d,h):
        super().__init__()
        self.n1,self.n2 = nn.LayerNorm(d),nn.LayerNorm(d)
        self.att = Attention(d,h)
        self.mlp = nn.Sequential(nn.Linear(d,4*d),nn.GELU(),
                                 nn.Linear(4*d,d))

    def forward(self,x):
        x = x + self.att(self.n1(x))
        return x + self.mlp(self.n2(x))

# Decision Transformer
class DT(nn.Module):
    def __init__(self,d=32,h=2):
        super().__init__()
        self.r,self.s,self.a = [nn.Linear(1,d) for _ in range(3)]
        self.te = nn.Embedding(T,d)
        self.blocks = nn.Sequential(Block(d,h),Block(d,h))
        self.ln = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d,1),nn.Tanh())

    def forward(self,g,s,a,t):
        te = self.te(t)[None]
        x = torch.stack([self.r(g)+te,self.s(s)+te,
                         self.a(a)+te],2).flatten(1,2)
        x = self.ln(self.blocks(x)).reshape(g.size(0),-1,3,32)
        return self.head(x[:,:,1])

# Training
model = DT()
opt = torch.optim.AdamW(model.parameters(),lr=2e-3)

for ep in range(20):
    p = torch.randperm(N)
    for i in range(0,N,256):
        b = p[i:i+256]
        loss = F.mse_loss(model(G[b],S[b],A[b],time),A[b])
        opt.zero_grad()
        loss.backward()
        opt.step()
    if ep%5==0 or ep==19:
        print(f"epoch {ep:2d} loss {loss.item():.4f}")

# Inference
@torch.no_grad()
def rollout(target,E=200):
    model.eval()
    s = torch.empty(E,1).uniform_(-1,1)
    g = torch.full((E,1),target)
    total,Ss,As,Gs = torch.zeros(E),[],[],[]

    for t in range(T):
        Ss.append(s)
        Gs.append(g/SCALE)
        As.append(torch.zeros(E,1))
        a = model(torch.stack(Gs,1),torch.stack(Ss,1),
                  torch.stack(As,1),time[:t+1])[:,-1]
        As[-1] = a
        s = s + 0.2*a
        r = -s.abs()
        total += r.squeeze(1)
        g -= r
    return total.mean().item()

for target in [-8.,-3.,-0.5]:
    print(f"target {target:5.1f} -> achieved {rollout(target):.2f}")


epoch  0 loss 0.1479
epoch  5 loss 0.1274
epoch 10 loss 0.1167
epoch 15 loss 0.1132
epoch 19 loss 0.1046
target  -8.0 -> achieved -6.97
target  -3.0 -> achieved -1.81
target  -0.5 -> achieved -0.90
